# Yorùbá voice-query pipeline — final composition (Colab + Drive)

End-to-end deployment chain. Verified on a Colab T4 with the v4 fine-tune sitting on Drive. No repo clone, no FAISS, no Mistral.

```
audio.wav
  ──► M1  Whisper-yo v4 (from Drive)       → YO text (already diacritized)
        ──► M3  NLLB-200 yor → eng        → EN query
              ──► M4  Qwen2.5-1.5B        → EN answer
                    ──► M5a NLLB eng → yor → YO answer text
                          ──► M5b mms-tts-yor → response.wav
```

**Three lessons baked in from earlier iterations**:

1. **M2 (diacritic restoration) is dropped.** The v4 fine-tune already produces canonical diacritized Yorùbá; piping it through `Davlan/mT5_base_yoruba_adr` *degrades* the output (introduces artefacts like `kẹ́jọ́míkàsà`). Verified live 2026-06-15. If you ever need M2 again (e.g. M1 = mlx-whisper which doesn't diacritize), uncomment the M2 block — it's in the file.
2. **VRAM-managed stage-by-stage loading.** T4 has 15 GB. Loading M1+NLLB+M4+M5b together OOMs. Each stage loads, runs, frees before the next loads. Peak ~6 GB; runs comfortably on T4. On A100 you can skip the frees.
3. **M4 chat-template fix.** `apply_chat_template(return_tensors="pt")` returns `BatchEncoding` in modern transformers, not a tensor — `prompt.shape` AttributeErrors. The fix is `tokenize=False` then explicit tokenization. Baked into Step 6.

**Runtime**: T4 fine. ~6–8 min total wall-clock the first time (Drive streams + model downloads); subsequent runs ~30 s for the actual inference.

## Step 1 — Auth + Drive mount

Only prompt cell. Authorize Drive in the popup, walk away.

In [ ]:
import os
from pathlib import Path

from google.colab import drive
try:
    drive.mount("/content/drive", force_remount=False)
except ValueError:
    drive.mount("/content/drive", force_remount=True)

DRIVE_TRAINING = Path("/content/drive/MyDrive/yoruba-pipeline-logs/training")
print(f"Drive mounted → {DRIVE_TRAINING}")

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF auth: ok")
else:
    print("HF auth: skipped — public models still work.")

## Step 2 — Install dependencies

In [ ]:
%%capture
!pip install -q "transformers>=4.45" "datasets>=3.0" "huggingface_hub>=0.24" \
    librosa soundfile accelerate sentencepiece protobuf

## Step 3 — Config + input audio

`V4_PATH = None` → auto-discover the most recent v4 training run on Drive. `INPUT_AUDIO` defaults to a streamed FLEURS sample (downloaded on first run); flip `USE_UPLOAD = True` to upload your own audio instead.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32
print(f"device : {DEVICE} ({DTYPE})")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# --- v4 fine-tune from Drive (None = auto-pick latest) ---
V4_PATH = None
if V4_PATH is None:
    runs = sorted(
        [p for p in DRIVE_TRAINING.iterdir() if (p / "merged_16bit").exists()],
        reverse=True,
    ) if DRIVE_TRAINING.exists() else []
    if not runs:
        raise FileNotFoundError(
            f"No run with merged_16bit/ under {DRIVE_TRAINING}. "
            "Run Whisper_v4.ipynb to completion first."
        )
    V4_PATH = str(runs[0] / "merged_16bit")
    V4_RUN  = runs[0].name
else:
    V4_RUN = Path(V4_PATH).parent.name
print(f"V4 run : {V4_RUN}")

# --- Input audio ---
INPUT_AUDIO  = Path("/content/sample.wav")
OUTPUT_AUDIO = Path("/content/response.wav")
USE_UPLOAD   = False    # True → upload your own audio in the next cell

REFERENCE_TEXT = ""
if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    fname = next(iter(uploaded))
    INPUT_AUDIO = Path("/content") / fname
    print(f"input  : {INPUT_AUDIO} (uploaded)")
else:
    if not INPUT_AUDIO.exists():
        print("Fetching one Yorùbá FLEURS sample (one-time)…")
        from datasets import load_dataset, Audio
        import soundfile as sf
        fleurs = load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
        sample = next(iter(fleurs.cast_column("audio", Audio(sampling_rate=16000))))
        sf.write(str(INPUT_AUDIO), sample["audio"]["array"], 16000)
        REFERENCE_TEXT = sample.get("raw_transcription") or ""
    print(f"input  : {INPUT_AUDIO}")
    if REFERENCE_TEXT:
        print(f"REF    : {REFERENCE_TEXT}")

# Tracking — final outputs from each stage
yo_raw = yo_diacritized = en_query = en_answer = yo_answer = None

## Step 4 — M1 ASR (audio → diacritized Yorùbá)

Loads the v4 fine-tune from Drive, transcribes the input audio, **frees the model** when done so M3/M4 fit alongside.

First load streams ~3 GB from Drive (~2–4 min on first run; cached for re-runs in the same session).

In [ ]:
import gc
import librosa
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

print(f"loading M1 from {V4_PATH}…")
m1_proc = WhisperProcessor.from_pretrained(V4_PATH)
m1_mdl  = WhisperForConditionalGeneration.from_pretrained(
    V4_PATH, torch_dtype=DTYPE,
).to(DEVICE).eval()
m1_mdl.generation_config.language = "<|yo|>"
m1_mdl.generation_config.task = "transcribe"
m1_mdl.generation_config.forced_decoder_ids = None

audio, _ = librosa.load(str(INPUT_AUDIO), sr=16000, mono=True)
feats = m1_proc.feature_extractor(
    audio, sampling_rate=16000, return_tensors="pt",
).input_features.to(DEVICE, dtype=DTYPE)
with torch.inference_mode():
    ids = m1_mdl.generate(
        feats, language="<|yo|>", task="transcribe",
        max_new_tokens=256, num_beams=1,
    )
yo_raw = m1_proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0].strip()

print(f"\nM1 → YO text:")
print(f"  {yo_raw}")
display(Audio(str(INPUT_AUDIO)))

# Free VRAM — M1 done, won't be needed again this run
del m1_mdl, m1_proc
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## (M2 — Diacritic restoration — **skipped by default**)

v4 already produces canonical diacritized Yorùbá. Running its output through `Davlan/mT5_base_yoruba_adr` introduces artefacts (verified 2026-06-15). The diacritic-restoration safety-net argument from earlier docs no longer applies when M1=v4.

If you ever switch M1 back to a non-diacritizing model (e.g. mlx-whisper), uncomment the cell below.

In [ ]:
# --- Optional M2: uncomment if M1 doesn't diacritize ---
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# m2_tok = AutoTokenizer.from_pretrained("Davlan/mT5_base_yoruba_adr")
# m2_mdl = AutoModelForSeq2SeqLM.from_pretrained("Davlan/mT5_base_yoruba_adr").to(DEVICE).eval()
# inputs = m2_tok(yo_raw, return_tensors="pt", truncation=True).to(DEVICE)
# with torch.inference_mode():
#     out = m2_mdl.generate(**inputs, max_new_tokens=256, num_beams=4)
# yo_diacritized = m2_tok.decode(out[0], skip_special_tokens=True).strip()
# del m2_mdl, m2_tok; gc.collect(); torch.cuda.empty_cache()
# print("M2 → YO diacritized:", yo_diacritized)

# Default: pass yo_raw straight through (already diacritized by v4)
yo_diacritized = yo_raw
print(f"M2 skipped — using v4's output directly:\n  {yo_diacritized}")

## Step 5 — Load NLLB (used by M3 and M5a)

NLLB-200 distilled-600M, loaded once and used for both translation legs (YO→EN here, EN→YO later). Stays in VRAM until after M5a.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

NLLB_ID = "facebook/nllb-200-distilled-600M"
print(f"loading NLLB from {NLLB_ID}…")
nllb_mdl = AutoModelForSeq2SeqLM.from_pretrained(NLLB_ID).to(DEVICE).eval()
print("  NLLB ready")

## Step 6 — M3 Translate (YO → EN)

In [ ]:
m3_tok = AutoTokenizer.from_pretrained(NLLB_ID, src_lang="yor_Latn")
inputs = m3_tok(yo_diacritized, return_tensors="pt", truncation=True).to(DEVICE)
with torch.inference_mode():
    out = nllb_mdl.generate(
        **inputs,
        forced_bos_token_id=m3_tok.convert_tokens_to_ids("eng_Latn"),
        max_new_tokens=256, num_beams=4,
        # NLLB occasionally loops on certain tokens (verified 2026-06-15 with
        # "...naturally, naturally, naturally..."). These two guard against it
        # without affecting healthy translations.
        no_repeat_ngram_size=3,
        repetition_penalty=1.15,
    )
en_query = m3_tok.batch_decode(out, skip_special_tokens=True)[0].strip()
print(f"M3 → EN query:\n  {en_query}")

## Step 7 — M4 Small LLM (EN query → EN answer)

Qwen2.5-1.5B-Instruct. **The chat-template fix is baked in**: render template to a string, then tokenize separately — avoids the `BatchEncoding has no .shape` error in modern transformers. Frees the model after answering.

In [ ]:
from transformers import AutoModelForCausalLM

M4_ID = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"loading M4 from {M4_ID}…")
m4_tok = AutoTokenizer.from_pretrained(M4_ID)
m4_mdl = AutoModelForCausalLM.from_pretrained(
    M4_ID, torch_dtype="auto", device_map="auto",
).eval()

messages = [
    {"role": "system",
     "content": "You are a concise assistant. Answer in 2-3 short sentences."},
    {"role": "user", "content": en_query},
]
# THE FIX: tokenize=False, then explicit tokenization
prompt_text = m4_tok.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False,
)
inputs = m4_tok(prompt_text, return_tensors="pt").to(m4_mdl.device)

with torch.inference_mode():
    out = m4_mdl.generate(
        **inputs, max_new_tokens=200, do_sample=False,
        pad_token_id=m4_tok.eos_token_id,
    )
input_len = inputs.input_ids.shape[-1]
en_answer = m4_tok.decode(out[0, input_len:], skip_special_tokens=True).strip()

print(f"\nM4 → EN answer:\n  {en_answer}")

# Free M4 — Qwen is large; M5b TTS still needs VRAM
del m4_mdl, m4_tok
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Step 8 — M5a Translate (EN → YO)

Reuses the NLLB model loaded in Step 5 with a different `src_lang` tokenizer. Frees NLLB after.

In [ ]:
m5a_tok = AutoTokenizer.from_pretrained(NLLB_ID, src_lang="eng_Latn")
inputs = m5a_tok(en_answer, return_tensors="pt", truncation=True).to(DEVICE)
with torch.inference_mode():
    out = nllb_mdl.generate(
        **inputs,
        forced_bos_token_id=m5a_tok.convert_tokens_to_ids("yor_Latn"),
        max_new_tokens=256, num_beams=4,
        # Same NLLB repetition guard as M3 (see comment there)
        no_repeat_ngram_size=3,
        repetition_penalty=1.15,
    )
yo_answer = m5a_tok.batch_decode(out, skip_special_tokens=True)[0].strip()
print(f"M5a → YO answer:\n  {yo_answer}")

# Free NLLB — both translation legs done
del nllb_mdl, m3_tok, m5a_tok
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Step 9 — M5b TTS (YO text → audio)

`facebook/mms-tts-yor` (VITS). Small (~0.5 GB), writes to `OUTPUT_AUDIO`, plays inline.

In [ ]:
import soundfile as sf
from transformers import VitsModel

M5_TTS_ID = "facebook/mms-tts-yor"
print(f"loading M5b TTS from {M5_TTS_ID}…")
tts_tok = AutoTokenizer.from_pretrained(M5_TTS_ID)
tts_mdl = VitsModel.from_pretrained(M5_TTS_ID).to(DEVICE).eval()

inputs = tts_tok(yo_answer, return_tensors="pt").to(DEVICE)
with torch.inference_mode():
    waveform = tts_mdl(**inputs).waveform.cpu().squeeze().numpy()

sr_out = tts_mdl.config.sampling_rate
sf.write(str(OUTPUT_AUDIO), waveform, sr_out)
print(f"\nM5b → {OUTPUT_AUDIO}  ({len(waveform)/sr_out:.2f}s @ {sr_out} Hz)")
display(Audio(str(OUTPUT_AUDIO)))

del tts_mdl, tts_tok
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Step 10 — Full-chain summary

Prints every stage's input/output side by side. Plays the input audio and the synthesized response back-to-back so you can listen for end-to-end quality.

In [ ]:
bar = "=" * 72
print(bar)
print(f"v4 run : {V4_RUN}")
print(f"INPUT  : {INPUT_AUDIO}")
if REFERENCE_TEXT:
    print(f"REF    : {REFERENCE_TEXT}")
print(bar)
print(f"M1 YO  : {yo_raw}")
print(f"M3 EN  : {en_query}")
print(f"M4 EN  : {en_answer}")
print(f"M5a YO : {yo_answer}")
print(f"M5b WAV: {OUTPUT_AUDIO}")
print(bar)

print("\nInput audio:")
display(Audio(str(INPUT_AUDIO)))
print("Synthesized response:")
display(Audio(str(OUTPUT_AUDIO)))

## Re-run on a new audio file

The models are already freed after each stage, so re-running is the same cost as the first run. To test new audio:

1. Flip `USE_UPLOAD = True` in Step 3 and re-run Step 3 to upload.
2. Run Steps 4 → 10 in order.

To test with a different v4 run, change `V4_PATH = "..."` in Step 3 to the specific run folder.